# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: 4 - Logistic Regression
Reason: Handling Multi-class classification task lane as my final output - 'decline_Score' range from (0-5)

**1.1: Import libraries**

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix

In [2]:
import pandas as pd
import os, getpass
import duckdb

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Time-aware split: latest 90 days window with 60 days training and 30 days testing rows. The training data will not have any client_id or content_id for training but will be included in testing data for testing and evaluation.

**2.1: Read data**

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data = pd.read_csv("baseline_action_score_ff.csv")

/tmp/ipykernel_659/4197380966.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("baseline_action_score_ff.csv")


In [4]:
data.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
0,2026-05-30,client_1a8bf67cad4ee525,content_5eb9c0b1de0202d2,9,17.444444,0.0,0.0,0.0,35,11-20,NaN,5
1,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,30.304348,0.0,0.0,0.0,131,21-100,NaN,5
2,2026-05-30,client_1a8bf67cad4ee525,content_5eb560fef36a11e6,1,11.000000,0.0,0.0,0.0,35,11-20,NaN,5
3,2026-05-30,client_1a8bf67cad4ee525,content_2c488c733b8c2220,15,11.666667,0.0,0.0,0.0,35,11-20,NaN,5
4,2026-03-01,client_c182d11e4862a37d,content_da76e1818babb4ce,132,11.348485,0.0,0.0,0.0,47,11-20,NaN,5
5,2026-03-01,client_c182d11e4862a37d,content_f4a0e5c90b283626,250,19.448000,0.0,0.0,0.0,47,11-20,NaN,5
6,2026-03-01,client_c182d11e4862a37d,content_d926564dfe83536b,47,31.595745,0.0,0.0,0.0,47,21-100,NaN,5
7,2026-04-16,client_23a62021009f63c4,content_02f32646ff66bb41,5,20.400000,0.0,0.0,0.0,47,21-100,NaN,5
8,2026-05-30,client_1a8bf67cad4ee525,content_92af9b4ca5a68926,16,11.687500,0.0,0.0,0.0,47,11-20,NaN,5
9,2026-04-16,client_23a62021009f63c4,content_e3df741ab59b92ab,84,29.476190,0.0,0.0,0.0,47,21-100,NaN,5


In [5]:
data.tail(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1475953,2026-04-19,client_e547b89c05043229,content_8ecaf32d7a6528e9,246,4.317073,0.008130,0.500000,0.500000,20,4-10,medium,0
1475954,2026-03-05,client_e547b89c05043229,content_33b343f69f0993df,307,4.433225,0.006515,0.500000,0.500000,24,4-10,medium,0
1475955,2026-03-23,client_fef1a8f436438636,content_5273b04321fb85d2,284,7.735915,0.003521,1.000000,0.500000,4,4-10,very_high,0
1475956,2026-04-16,client_23a62021009f63c4,content_d8a9f27682068c84,85,8.917647,0.011765,0.014378,0.007003,12,4-10,low,0
1475957,2026-04-16,client_23a62021009f63c4,content_92a8d1dac17a165a,832,9.338942,0.002404,0.333333,0.166667,0,4-10,medium,0
1475958,2026-03-02,client_e547b89c05043229,content_3508adb0f05ec0b9,2412,2.861526,0.009950,0.041667,0.375000,24,1-3,low,0
1475959,2026-03-02,client_e547b89c05043229,content_cf8a731286a0394b,519,1.967245,0.009634,0.666667,0.666667,24,1-3,high,0
1475960,2026-03-02,client_e547b89c05043229,content_5538d1fc8d19fa4a,631,3.606973,0.004754,0.500000,0.500000,10,4-10,medium,0
1475961,2026-03-12,client_e5c2aa26a8598242,content_af5d858edcd9e4f1,161,5.161491,0.006211,0.500000,0.333333,25,4-10,medium,0
1475962,2026-04-16,client_23a62021009f63c4,content_cad4a46da4671397,574,5.498258,0.008711,0.100000,0.038462,10,4-10,low,0


**2.2: Handle missing values**

In [6]:
data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update', 'position_bucket', 'engagement_bucket',
       'decline_score'],
      dtype='object')

In [7]:
data['gsc_avg_position'].isnull().sum()

np.int64(8)

In [8]:
data['ctr'].isnull().sum()

np.int64(0)

In [9]:
data['engagement_rate'].isnull().sum()

np.int64(20536)

In [10]:
data['scroll_rate'].isnull().sum()

np.int64(2595)

In [11]:
data['days_since_update'].isnull().sum()

np.int64(0)

In [12]:
data['decline_score'].isnull().sum()

np.int64(0)

In [13]:
data = data.dropna(subset=['gsc_avg_position']).reset_index(drop=True)

In [14]:
data = data.dropna(subset=['engagement_rate']).reset_index(drop=True)

In [15]:
data = data.dropna(subset=['scroll_rate']).reset_index(drop=True)

In [16]:
data['gsc_avg_position'].isnull().sum()

np.int64(0)

In [17]:
data['engagement_rate'].isnull().sum()

np.int64(0)

In [18]:
data['scroll_rate'].isnull().sum()

np.int64(0)

**2.3: Train-Test split**

In [19]:
data['report_date'] = pd.to_datetime(data['report_date'])

train_data = data[(data['report_date'] >= '2026-03-01') & (data['report_date'] <= '2026-04-30')].copy()

test_data = data[(data['report_date'] >= '2026-05-01') & (data['report_date'] <= '2026-05-30')].copy()

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")
display(train_data.head())
display(test_data.head())

Training set size: 817242
Testing set size: 635596


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,30.304348,0.0,0.0,0.0,131,21-100,NaN,5
4,2026-03-01,client_c182d11e4862a37d,content_da76e1818babb4ce,132,11.348485,0.0,0.0,0.0,47,11-20,NaN,5
5,2026-03-01,client_c182d11e4862a37d,content_f4a0e5c90b283626,250,19.448000,0.0,0.0,0.0,47,11-20,NaN,5
6,2026-03-01,client_c182d11e4862a37d,content_d926564dfe83536b,47,31.595745,0.0,0.0,0.0,47,21-100,NaN,5
7,2026-04-16,client_23a62021009f63c4,content_02f32646ff66bb41,5,20.400000,0.0,0.0,0.0,47,21-100,NaN,5


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
0,2026-05-30,client_1a8bf67cad4ee525,content_5eb9c0b1de0202d2,9,17.444444,0.0,0.0,0.0,35,11-20,NaN,5
2,2026-05-30,client_1a8bf67cad4ee525,content_5eb560fef36a11e6,1,11.000000,0.0,0.0,0.0,35,11-20,NaN,5
3,2026-05-30,client_1a8bf67cad4ee525,content_2c488c733b8c2220,15,11.666667,0.0,0.0,0.0,35,11-20,NaN,5
8,2026-05-30,client_1a8bf67cad4ee525,content_92af9b4ca5a68926,16,11.687500,0.0,0.0,0.0,47,11-20,NaN,5
22,2026-05-30,client_a22068e339bf95f5,content_6c7f93a26b1feb6c,35,14.228571,0.0,0.0,0.0,35,11-20,NaN,5


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**3.1: Train X-y split**

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_train = train_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id','position_bucket', 'engagement_bucket', 'decline_score'], axis=1)
y_train = train_data['decline_score']

In [21]:
X_train.columns

Index(['gsc_impressions', 'gsc_avg_position', 'ctr', 'engagement_rate',
       'scroll_rate', 'days_since_update'],
      dtype='object')

In [22]:
print(len(X_train))
print(len(y_train))

817242
817242


**3.2: Test X-y split**

In [23]:
X_test = test_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id','position_bucket', 'engagement_bucket','decline_score'], axis=1)
y_test = test_data['decline_score']

In [24]:
X_test.columns

Index(['gsc_impressions', 'gsc_avg_position', 'ctr', 'engagement_rate',
       'scroll_rate', 'days_since_update'],
      dtype='object')

In [25]:
print(len(X_test))
print(len(y_test))

635596
635596


**3.3: Model: LogisticRegression**

**3.3.1: Model training**

In [26]:
model1 = LogisticRegression(max_iter=1000, class_weight='balanced', multi_class='multinomial')
model1.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000,
                   multi_class='multinomial')

**3.3.2: Model predictions and evaluation**

In [27]:
y_pred = model1.predict(X_test)
y_proba = model1.predict_proba(X_test)


In [28]:
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba, multi_class='ovo')

print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

F1: 0.4740
ROC-AUC: 0.8702


**3.3.3: Comparison table**

In [29]:
print("Actual (y_test) vs Predicted (y_pred):")
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
display(comparison_df.head(20))
display(comparison_df.tail(20))
print(f"Total rows: {len(comparison_df)}\n")

Actual (y_test) vs Predicted (y_pred):


,Actual,Predicted
0,5,4
2,5,4
3,5,4
8,5,4
22,5,4
23,5,5
24,5,4
38,5,4
39,5,4
40,5,5


,Actual,Predicted
1452735,0,1
1452737,0,2
1452739,0,2
1452741,0,2
1452747,0,0
1452748,0,2
1452750,0,2
1452756,0,0
1452757,0,0
1452758,0,1


Total rows: 635596



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

**4.1: Confusion matrix**

In [31]:
print(confusion_matrix(y_test, y_pred))

[[ 7307  1591  1409     0     0     0]
 [ 6434 10352  3269  1729  1196   245]
 [ 8666 16581 79917  5735  3017  1572]
 [  555 12992 58407 66673 32157 12602]
 [    2  2521 12391 48041 66789 44980]
 [    0     9     2  1574 36828 90053]]


**4.2: Classification report**

In [32]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.32      0.71      0.44     10307
           1       0.24      0.45      0.31     23225
           2       0.51      0.69      0.59    115488
           3       0.54      0.36      0.43    183386
           4       0.48      0.38      0.42    174724
           5       0.60      0.70      0.65    128466

    accuracy                           0.51    635596
   macro avg       0.45      0.55      0.47    635596
weighted avg       0.52      0.51      0.50    635596



**Interpretation**

The classifier shows an asymmetric precision-recall pattern that varies by class position, not a uniform extremes-vs-middle split. Class 0 has high recall (71%) but low precision (32%): the model over-predicts this class, catching most true 0s but misfiring on cases that belong to classes 1 and 2. Class 5, by contrast, now performs strongly on both fronts, recall 70% and precision 60%, making it the best-separated class in the matrix, with little bleed from neighboring classes. Classes 3 and 4 show a mixed pattern: moderate precision (54%, 48%) but weak recall (36%, 38%), meaning predictions for these classes are reasonably trustworthy, but most true cases get pulled away — class 3's true instances split heavily into classes 2 (58,407) and 4 (32,157), and class 4's into classes 3 (48,041) and 5 (44,980). Class 1 is the weakest class overall, with both low precision (24%) and low recall (45%): it absorbs heavy misclassification from classes 0 and 2, and its own true cases scatter across 0, 2, and 3 rather than concentrating.

Support remains heavily skewed toward classes 2-4 (473,598 of 635,596 rows), so the 0.51 accuracy and 0.50 weighted F1 are dominated by mid-range performance, while the 0.47 macro F1 (matching the reported 0.4740) shows that once class size is ignored, class 1's weakness and class 0's imprecision drag the average down noticeably.

Combined with the confusion matrix, this indicates the model still treats decline severity as an ordinal continuum it can rank (consistent with the 0.87 ROC-AUC), with most errors landing on adjacent classes but the boundary at class 5 is now well-resolved, while class 1 (and to a lesser extent class 0) remains the hardest region to separate from its neighbors.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.